In [1]:
import pandas as pd
from transformers import (
    T5Tokenizer,
    Trainer,
    TrainingArguments,
    T5ForConditionalGeneration
)

C:\Users\Suraj Verma\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [3]:
train_data.shape


(14732, 3)

In [4]:
val_data.shape

(818, 3)

In [5]:
# Random Sampling
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)

# Data Pre_Processing

In [6]:
import re
def clean_data(text):
    text=re.sub(r"\r\n"," ",text) # line
    text=re.sub(r"\s+"," ",text)  #spaces
    text=re.sub(r"<.*?>"," ",text) # html tags <p1> <h1>
    text=text.strip().lower()
    return text   
    

In [7]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=train_data["dialogue"].apply(clean_data)
val_data["summary"]=train_data["summary"].apply(clean_data)

# Tokenize

In [8]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [9]:
# raw data => tokenize inputs for fine-tunning
def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncate=True)

    inputs["labels"]=targets["input_ids"] # token ids => add to input as labels
    return inputs

In [10]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [11]:
# input Ids- dialogue=> token ids
# 1 => EOS, 0=> padding
# attention mask
#labels-target=>summary token

# Working with our model

In [12]:
# NLP => generation task
model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1082.01it/s]


In [13]:
import torch
if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")
print(f"device: {device}")
model.to(device)
    

device: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [14]:
# trainign Arguements
training_args=TrainingArguments(
    output_dir="./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0=> lr default
)

In [15]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [16]:
trainer.train()

NameError: name 'tr' is not defined

In [ ]:
#model load => fine-tune => save the model

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

In [ ]:
mode.from_pretrained("./saved_summary_model")
tokenizer.from_pretrained("./saved_summary_model")